In [ ]:
!pip install transformers

In [ ]:
!pip install "transformers[torch]"

In [4]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data.shape

(14732, 3)

In [5]:
val_data.shape

(818, 3)

In [6]:
# random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

Data Pre-Processing

In [7]:
import re

def clean_data(text):
  text = re.sub(r"\r\n", " ", text) # lines
  text = re.sub(r"\s+", " ", text) # spaces
  text = re.sub(r"<.*?>", " ", text) # html tags
  text = text.strip().lower()
  return text

In [8]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [9]:
train_data["dialogue"][10]

'mom: where are you? anne: i am getting my blood tested! mom: for your acne? anne: yes : ( mom: poor you mom: i will cook something delicious for you anne: thanks mom! anne: i will be tonight then! mom: be waiting at home! 😚'

Tokenize

In [10]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [11]:
# raw data ==> tokenized inputs for fine-tuning

def tokenize(data):
  inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
  targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

  inputs["labels"] = targets["input_ids"] # token ids => add to input as labels
  return inputs

In [12]:
train_dataset = train_data.apply(tokenize, axis=1).tolist() # axis=1 ==> Apply this function one row at a time.
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [13]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [14]:
# input ids - dialogue => token ids
# 1 => EOS, 0 => padding

# attention mask
# 1 => token_id is valid, 0 => not valid

# labels - target => summary token

# Working with the model

In [15]:
# NLP => generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

#Fine-tune

In [12]:
from torch.distributed import is_available
import torch

if torch.backends.mps.is_available():
  device = torch.device("mps")
elif torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

print("device = ",device)

model.to(device)

device =  cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [17]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500,
    # 0 => lr default
)


In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [19]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.650621,0.382246
2,0.396779,0.359491
3,0.373510,0.354226
4,0.361317,0.349932
5,0.355290,0.348960
6,0.351093,0.348591


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9147684682210286, metrics={'train_runtime': 1321.5662, 'train_samples_per_second': 18.16, 'train_steps_per_second': 2.27, 'total_flos': 3248203235328000.0, 'train_loss': 0.9147684682210286, 'epoch': 6.0})

# Save Model

In [20]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [9]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

# Test the core logic for summarization

In [10]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue) # clean

  # tokenize
  inputs = tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt"
  ).to(device)

  # generate summary => token ids
  model.to(device)
  targets = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length=150,
      num_beams=4, # compare 4 differnt summaries and gives the best
      early_stopping=True
  )

  # token ids convert to summary => decoding
  summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, ...
  return summary

In [16]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence
continues to expand rapidly across industries, from healthcare
to finance and education. Recent reports suggest that AI adoption
has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning
systems to automate tasks, improve decision-making, and enhance
customer experiences. However, this growth has also raised
questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in
deep learning and access to large datasets. These models can now
perform complex tasks such as language understanding, image
recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in
AI models, as they often reflect the data they are trained on.
Ensuring fairness and transparency is becoming a key area of
research.

Reporter: Governments and organizations are beginning to introduce
regulations to guide the development and deployment of AI
technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems,
especially deep neural networks, operate as black boxes, making it
difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI
development, including data privacy, security, and long-term
societal impact.

Expert: Looking ahead, collaboration between researchers,
policymakers, and industry leaders will be crucial to ensure that
AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)
print("Summary: ", summary)

Summary:  ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.
